In [1]:
%load_ext autoreload
%autoreload 2

import nglview as nv
import matplotlib.pyplot as plt
import numpy as np
import MDAnalysis as mda 
from MDAnalysis.transformations.translate import translate
from IPython.core.display import Image
from numpy import random
from scipy.stats import binom
import os
from MDAnalysis.analysis import align
from MDAnalysis.coordinates.memory import MemoryReader
from MDAnalysis.analysis import distances
import param_tool as pt

In [4]:
PTM_folder = 'AF_647_cys'
base_name = '647'


In [5]:
mol_path = f'{PTM_folder}/molecules/substructure/AF_647_no_H.pdb'
GGG = mda.Universe(f'molecules/ACE-GGG-NME.pdb')
PTM = mda.Universe(mol_path)

view = nv.show_mdanalysis(PTM)
view.clear()
view.add_ball_and_stick()
view
# nv.show_mdanalysis(PTM)

/trinity/home/n_kristovsky/.conda/envs/d_topmol/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:479: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn(


NGLWidget()

In [54]:
# 1. Удаляем все протоны в KMA
# Выбираем атомы, которые НЕ являются водородами (тип H)
# no_hydrogens = PTM.select_atoms("not type H")
# PTM = no_hydrogens.atoms
# PTM.residues.resids = 2
# PTM.residues.resnums = 2
# PTM.residues.resnames = base_name

In [6]:
atom_names = ["C", "CA", "N"] 
atom_align_names_list =[f'resnum 2 and name {name}' for name in atom_names] 
mobile_atom_list = [f'name {name}' for name in atom_names] 

model1_model2_align = align.alignto(PTM, GGG ,{'mobile':atom_align_names_list, 'reference':atom_align_names_list}) 
#align model1 and model2

G_PTM_G = mda.Merge(GGG.select_atoms('resnum 0:1'), PTM.select_atoms('resnum 2'), GGG.select_atoms('resnum 3:4'))

In [7]:
view = nv.show_mdanalysis(G_PTM_G)
view.clear()
view.add_ball_and_stick()
view

NGLWidget()

In [8]:
G_PTM_G.atoms.write(f"molecules/tripeptides/ACE_G_{PTM.residues.resnames[0]}_G_NME.pdb")

/trinity/home/n_kristovsky/.conda/envs/d_topmol/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/trinity/home/n_kristovsky/.conda/envs/d_topmol/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


In [9]:
# from MDAnalysis.analysis import distances
import numpy as np

# Рассчитываем все попарные расстояния между атомами
dist_matrix = distances.distance_array(G_PTM_G.atoms.positions, 
                                      G_PTM_G.atoms.positions)

# Находим индексы пар атомов с расстоянием < 1.0 Å (исключая диагональ)
i, j = np.where((dist_matrix < 1.0) & (dist_matrix > 0))

if len(i) > 0:
    print(f"Найдено {len(i)} клишей!")
    for a, b in zip(i, j):
        # Убираем дубликаты (i-j и j-i)
        vec = G_PTM_G.atoms[b].position - G_PTM_G.atoms[a].position
        G_PTM_G.atoms[b].position += 0.3 * vec / np.linalg.norm(vec)
        if a < b:
            atom1 = G_PTM_G.atoms[a]
            atom2 = G_PTM_G.atoms[b]
            print(f"Клиш: {atom1.resname}{atom1.resid}:{atom1.name} <-> "
                  f"{atom2.resname}{atom2.resid}:{atom2.name} | "
                  f"Расстояние: {dist_matrix[a,b]:.2f} Å")
        
else:
    print("Клэшей нет!")

Клэшей нет!


In [30]:
pt.file_opener(mol_path, format_coord='3D')

[14:51:54] Explicit valence for atom # 24 O, 3, is greater than permitted


AtomValenceException: Explicit valence for atom # 24 O, 3, is greater than permitted

In [20]:
view = nv.show_mdanalysis(G_PTM_G)
view.clear()
view.add_ball_and_stick()
view

NGLWidget()

In [22]:
G_PTM_G.residues.segids

array(['A', 'A', 'A', 'A', 'A'], dtype=object)

In [ ]:
Chem.MolFrom

In [177]:
PTM_name = 'KMA'
G_PTM_G.atoms.write(f"PDB/G_{PTM_name}_RN_G.pdb")

In [169]:
G_KMA_G = mda.Universe('PDB/G_KMA_H_G.pdb')
nv.show_mdanalysis(G_KMA_G)

NGLWidget()